In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker, read_sparse_h5, recompute_aggregate_scores

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Human_hipp/")

Prepare data

In [3]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")

In [4]:
(rna.obs_names == atac.obs_names).all()

np.True_

In [5]:
atac.obsm["spatial"] = rna.obsm["spatial"].copy()

In [6]:
# import anndata as ad
# import h5py
# import numpy as np
# from scipy import sparse

# def h5ad_to_h5(adata, output_file: str):

#     if adata.raw is not None and adata.raw.X is not None:
#         X = adata.raw.X
#         features = np.asarray(adata.raw.var_names, dtype=str)
#     else:
#         X = adata.X
#         features = np.asarray(adata.var_names, dtype=str)

#     barcodes = np.asarray(adata.obs_names, dtype=str)

#     spatial = None
#     if "spatial" in adata.obsm:
#         spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

#     if sparse.issparse(X):
#         X = X.tocsr()
#     else:
#         X = sparse.csr_matrix(X)

#     if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
#         X.data = X.data.astype(np.int32, copy=False)
#     else:
#         X.data = X.data.astype(np.float32, copy=False)

#     with h5py.File(output_file, "w") as f:
#         g = f.create_group("matrix")

#         g.create_dataset("data", data=X.data,
#                          compression="gzip", compression_opts=4, shuffle=True)

#         g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
#                          compression="gzip", compression_opts=4, shuffle=True)

#         g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
#                          compression="gzip", compression_opts=4, shuffle=True)

#         g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

#         g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
#         g.create_dataset("features", data=np.array(features, dtype="S"))

#         if spatial is not None:
#             g.create_dataset(
#                 "spatial",
#                 data=spatial,
#                 compression="gzip",
#                 compression_opts=4,
#                 shuffle=True
#             )

In [7]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(atac, output_file="atac.h5")

In [8]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [9]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Human_hipp/"
# bm.run(methods=["Single_modal"
#                ],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        n_cluster=7,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_hipp/",
#        hvg_num=3000,
#        )

In [10]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_hipp/atac.csv", index_col=0)

In [11]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [12]:
# sc.pl.umap(rna, color=["cluster", "cell_type"])
# sc.pl.spatial(rna, color="cluster", spot_size=1)

In [13]:
# methods =  ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",    "scMM",       "scMDC",
#                 "Matilda",   "moETM",  "MISO",   "SpatialGlue",  "COSMOS",     "PRESENT",
#                 "SMOPCA",       "CellCharter"
#                ]
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_hipp/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

Plot

In [14]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_hipp/"
methods = ["Seurat_WNN", "MOFA2", "MultiVI", "Multigrate", "scMM", "scMDC", "Matilda", "moETM",
"MISO", "SpatialGlue",  "COSMOS", "PRESENT", "SMOPCA", "CellCharter"]
res = bm.read_result(path=result_folder,
                     methods=methods + ["rna", "atac"],
                     reindex=False)

2026-04-08 22:47:33 - WARNING - '_latent' result for 'Seurat_WNN' not found at: /mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_hipp/seurat_wnn_latent.csv


In [15]:
res["Embed"]["Seurat_WNN"] = res["Embed"]["SpatialGlue"].copy()
res["seurat_wnn_conn"] = read_sparse_h5(f"{result_folder}/seurat_wnn_connection.h5")[0]
res["seurat_wnn_dist"] = read_sparse_h5(f"{result_folder}/seurat_wnn_distance.h5")[0]

In [16]:
rna.obs["batch"] = ["batch1"]*1000 + ["batch2"] * (rna.shape[0]-1000)

In [17]:
# metrics = bm.cal_metrics(adata=rna, batch_key="batch", label_key="cell_type",
#                          res_dict=res, methods="all", verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [18]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)

In [19]:
metric = metrics[0][['Isolated labels', 'NMI', 'ARI', 'Silhouette label',
       'cLISI', 'CHAOS', 'PAS', 'Domain continuity','Bio conservation']]

In [20]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Mouse_hipp"

In [21]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [22]:
metric = recompute_aggregate_scores(metric)

In [23]:
metric = metric.drop(["rna", "atac"])

In [24]:
metric.columns = ['Isolated labels', 'NMI', 'ARI', 'Silhouette label', 'cLISI', 'CHAOS',
       'PAS', 'Domain continuity', 'Bio conservation']

In [25]:
# bm.plot_heatmap(metric_df=metric, total_name="Bio conservation",
#                 save=f"{figure_save_dir}/summary_heatmap.pdf",
#                 show_top=7,
#                 show_bottom=0,
#                 insert_marker_row = 8,
#                 )

In [26]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [rna.obsm["spatial"]]
spatial = transform_coord(spatial, vertical=True, axis="y")

In [27]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"

In [28]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(1.97, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["RNA", "ATAC"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "atac"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_single_modal.pdf",
#                 rasterized=True
#                 )

In [29]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=6,
#                 xlabel=["Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True
#                 )

In [30]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(16, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=8,
#                 xlabel=["RNA", "ATAC", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "atac", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True
#                 )

In [31]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(rna.obs["cell_type"]).reshape(-1,1)},
#                 figsize=(1.97, 2.03),
#                 frameon=True,
#                 inner_gs_row=1, inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.015,
#                 save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [34]:
# bm.plot_legend(category_lst=["White matter", "Hilus", "GCL", "ML", "SLM", "Chroid plexus", "CA3 Pyr"],
#                 marker="o",
#                 ncol=2,
#                 # save=f"{figure_save_dir}/annot_legend.pdf",
#                 order=["CA3 Pyr", "Chroid plexus", "GCL", "White matter", "Hilus", "ML",  "SLM"]
#                 )

In [ ]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=None,
#              annot_list=list(rna.obs["cell_type"]),
#              figsize=(10, 6.7),
#              frameon=True,
#              inner_gs_row=1,
#              inner_gs_col=1,
#              size=15,
#              ncol=5,
#              xlabel=["RNA", "ATAC", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", 
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              only_show_top=False,
#              ylabel=None,
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["rna", "atac", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#             #  ylabel_pad=0.02,
#              xlabel_pad=0.014,
#              outer_row_hspace=0.22,
#              merge=False,
#              merge_margin_size=0.4,
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [67]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=None,
#              annot_list=list(rna.obs["cell_type"]),
#              figsize=(16, 4.5),
#              frameon=True,
#              inner_gs_row=1,
#              inner_gs_col=1,
#              size=10,
#              ncol=8,
#              xlabel=["RNA","ATAC", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              only_show_top=False,
#              ylabel=None,
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["rna","atac", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.02,
#              save_dpi=600,
#             #  ylabel_pad=0.02,
#              xlabel_pad=0.012,
#              outer_row_hspace=0.22,
#              merge=False,
#              merge_margin_size=0.4,
#              save=f"{figure_save_dir}/umap_methods_all.pdf"
# )